# 문자 단위 RNN(Char RNN)
#### RNN의 입출력의 단위가 단어 레벨(word-level)이 아니라 문자 레벨(character-level)로 하여 RNN을 구현한다면, 이를 문자 단위 RNN이라고 함. RNN 구조 자체가 달라진 것은 아니고, 입, 출력의 단위가 문자로 바뀌었을 뿐. 문자 단위 RNN을 다대다 구조로 구현하고자 함



## 1. 문자 단위 RNN(Char RNN)

In [1]:
# 도구들 임포트
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

### 1-1) 훈련 데이터 전처리하기
- 여기서는 문자 시퀀스 apple을 입력받으면 pple!를 출력하는 RNN을 구현. 이렇게 구현하는 어떤 의미가 있지는 않음. 그저 RNN의 동작을 이해하기 위한 목적
- 입력 데이터와 레이블 데이터에 대해서 문자 집합(voabulary)을 만듦. 여기서 문자 집합은 중복을 제거한 문자들의 집합





In [2]:
input_str = 'apple'
label_str = 'pple!'
char_vocab = sorted(list(set(input_str+label_str)))
vocab_size = len(char_vocab)
print ('문자 집합의 크기 : {}'.format(vocab_size))

문자 집합의 크기 : 5


- 현재 문자 집합에는 총 5개의 문자(!, a, e, l, p)가 존재
- 이제 하이퍼파라미터를 정의해주기. 이때 입력은 원-핫 벡터를 사용할 것이므로 입력의 크기는 문자 집합의 크기여야만 함

In [3]:
input_size = vocab_size # 입력의 크기는 문자 집합의 크기
hidden_size = 5
output_size = 5
learning_rate = 0.1

In [4]:
# 문자 집팝에 고유한 정수 부여하기

char_to_index = dict((c, i) for i, c in enumerate(char_vocab)) # 문자에 고유한 정수 인덱스 부여
print(char_to_index)

{'!': 0, 'a': 1, 'e': 2, 'l': 3, 'p': 4}


- !은 0, a는 1, e는 2, l은 3, p는 4가 부여되었음. 나중에 예측 결과를 다시 문자 시퀀스로 보기위해서 반대로 정수로부터 문자를 얻을 수 있는 index_to_char을 만듦



In [5]:
index_to_char={}
for key, value in char_to_index.items():
    index_to_char[value] = key
print(index_to_char)

{0: '!', 1: 'a', 2: 'e', 3: 'l', 4: 'p'}


In [6]:
# 입력 데이터와 레이블 데이터의 각 문자들을 정수로 맵핑

x_data = [char_to_index[c] for c in input_str]
y_data = [char_to_index[c] for c in label_str]
print(x_data)
print(y_data)

[1, 4, 4, 3, 2]
[4, 4, 3, 2, 0]


In [7]:
# 파이토치의 nn.RNN()은 기본적으로 3차원 텐서를 입력받기 때문에 배치 차원을 추가해주기

# 배치 차원 추가
# 텐서 연산인 unsqueeze(0)를 통해 해결할 수도 있었음.
x_data = [x_data]
y_data = [y_data]
print(x_data)
print(y_data)

[[1, 4, 4, 3, 2]]
[[4, 4, 3, 2, 0]]


In [8]:
# 입력 시퀀스의 각 문자들을 원-핫 벡터로 바꾸기

x_one_hot = [np.eye(vocab_size)[x] for x in x_data]
print(x_one_hot)

[array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.]])]


In [9]:
# 입력 데이터와 레이블 데이터를 텐서로 변경
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

/tmp/ipykernel_3456/899184407.py:2: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  X = torch.FloatTensor(x_one_hot)


In [10]:
# 텐서의 크기 확인
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터의 크기 : torch.Size([1, 5, 5])
레이블의 크기 : torch.Size([1, 5])


### 1-2) 모델 구현하기

In [11]:
class Net(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_size, hidden_size, batch_first=True) # RNN 셀 구현
        self.fc = torch.nn.Linear(hidden_size, output_size, bias=True) # 출력층 구현

    def forward(self, x): # 구현한 RNN 셀과 출력층을 연결
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

In [12]:
# 클래스로 정의한 모델을 net에 저장
net = Net(input_size, hidden_size, output_size)

In [13]:
# 출력 크기 확인
outputs = net(X)
print(outputs.shape) # 3차원 텐서

torch.Size([1, 5, 5])


- (1, 5, 5)의 크기를 가지는데 각각 배치 차원, 시점(timesteps), 출력의 크기
- 나중에 정확도를 측정할 때는 이를 모두 펼쳐서 계산하게 되는데, 이때는 view를 사용하여 배치 차원과 시점 차원을 하나로 만듦



In [14]:
print(outputs.view(-1, input_size).shape) # 2차원 텐서로 변환

torch.Size([5, 5])


In [16]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([1, 5])
torch.Size([5])


- 레이블 데이터는 (1, 5)의 크기를 가지는데, 마찬가지로 나중에 정확도를 측정할 때는 이걸 펼쳐서 계산할 예정
- 이 경우 (5)의 크기를 가지게 됨
- 이제 옵티마이저와 손실 함수를 정의



In [17]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [18]:
# 총 100번의 에포크 학습
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X)
    loss = criterion(outputs.view(-1, input_size), Y.view(-1)) # view를 하는 이유는 Batch 차원 제거를 위해
    loss.backward() # 기울기 계산
    optimizer.step() # 아까 optimizer 선언 시 넣어둔 파라미터 업데이트

    # 아래 세 줄은 모델이 실제 어떻게 예측했는지를 확인하기 위한 코드.
    result = outputs.data.numpy().argmax(axis=2) # 최종 예측값인 각 time-step 별 5차원 벡터에 대해서 가장 높은 값의 인덱스를 선택
    result_str = ''.join([index_to_char[c] for c in np.squeeze(result)])
    print(i, "loss: ", loss.item(), "prediction: ", result, "true Y: ", y_data, "prediction str: ", result_str)

0 loss:  1.5673366785049438 prediction:  [[4 4 4 4 4]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  ppppp
1 loss:  1.3584315776824951 prediction:  [[4 4 4 4 2]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  ppppe
2 loss:  1.1831518411636353 prediction:  [[4 4 4 4 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppp!
3 loss:  1.0074036121368408 prediction:  [[4 4 4 4 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppp!
4 loss:  0.8461812734603882 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
5 loss:  0.7129228115081787 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
6 loss:  0.6102717518806458 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
7 loss:  0.5304067134857178 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
8 loss:  0.466200053691864 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
9 loss:  0.4150741696357727 prediction:  [[4 4 4 2 0]] t

## 2. 더 많은 데이터로 학습한 문자 단위 RNN(Char RNN)

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim

#### 2-1) 훈련 데이터 전처리하기

In [20]:
sentence = ("if you want to build a ship, don't drum up people together to "
            "collect wood and don't assign them tasks and work, but rather "
            "teach them to long for the endless immensity of the sea.")

In [21]:
# 문자 집합 생성, 각 문자에 고유한 정수 부여

char_set = list(set(sentence)) # 중복을 제거한 문자 집합 생성
char_dic = {c: i for i, c in enumerate(char_set)} # 각 문자에 정수 인코딩

In [22]:
print(char_dic) # 공백도 여기서는 하나의 원소

{'m': 0, 't': 1, 'h': 2, 'c': 3, 'k': 4, 's': 5, 'o': 6, '.': 7, 'l': 8, ',': 9, 'a': 10, "'": 11, 'n': 12, 'g': 13, 'b': 14, 'y': 15, 'e': 16, 'r': 17, ' ': 18, 'p': 19, 'i': 20, 'd': 21, 'u': 22, 'w': 23, 'f': 24}


- 각 문자에 정수가 부여되었으며, 총 25개의 문자가 존재. 문자 집합의 크기를 확인하기

In [23]:
dic_size = len(char_dic)
print('문자 집합의 크기 : {}'.format(dic_size))

문자 집합의 크기 : 25


- 문자 집합의 크기는 25이며, 입력을 원-핫 벡터로 사용할 것이므로 이는 매 시점마다 들어갈 입력의 크기이기도 함. 이제 하이퍼파라미터를 설정
- hidden_size(은닉 상태의 크기)를 입력의 크기와 동일하게 줬는데, 이는 사용자의 선택으로 다른 값을 줘도 무방



In [24]:
# 하이퍼파라미터 설정
hidden_size = dic_size
sequence_length = 10  # 임의 숫자 지정
learning_rate = 0.1

In [25]:
# 데이터 구성
x_data = []
y_data = []

for i in range(0, len(sentence) - sequence_length):
    x_str = sentence[i:i + sequence_length]
    y_str = sentence[i + 1: i + sequence_length + 1]
    print(i, x_str, '->', y_str)

    x_data.append([char_dic[c] for c in x_str])  # x str to index
    y_data.append([char_dic[c] for c in y_str])  # y str to index

0 if you wan -> f you want
1 f you want ->  you want 
2  you want  -> you want t
3 you want t -> ou want to
4 ou want to -> u want to 
5 u want to  ->  want to b
6  want to b -> want to bu
7 want to bu -> ant to bui
8 ant to bui -> nt to buil
9 nt to buil -> t to build
10 t to build ->  to build 
11  to build  -> to build a
12 to build a -> o build a 
13 o build a  ->  build a s
14  build a s -> build a sh
15 build a sh -> uild a shi
16 uild a shi -> ild a ship
17 ild a ship -> ld a ship,
18 ld a ship, -> d a ship, 
19 d a ship,  ->  a ship, d
20  a ship, d -> a ship, do
21 a ship, do ->  ship, don
22  ship, don -> ship, don'
23 ship, don' -> hip, don't
24 hip, don't -> ip, don't 
25 ip, don't  -> p, don't d
26 p, don't d -> , don't dr
27 , don't dr ->  don't dru
28  don't dru -> don't drum
29 don't drum -> on't drum 
30 on't drum  -> n't drum u
31 n't drum u -> 't drum up
32 't drum up -> t drum up 
33 t drum up  ->  drum up p
34  drum up p -> drum up pe
35 drum up pe -> rum up peo
36

- 총 170개의 샘플이 생성되었음. 그리고 각 샘플의 각 문자들은 고유한 정수로 인코딩이 된 상태
- 첫번째 샘플의 입력 데이터와 레이블 데이터를 출력하기



In [26]:
print(x_data[0])
print(y_data[0])

[20, 24, 18, 15, 6, 22, 18, 23, 10, 12]
[24, 18, 15, 6, 22, 18, 23, 10, 12, 1]


In [27]:
x_one_hot = [np.eye(dic_size)[x] for x in x_data] # x 데이터는 원-핫 인코딩
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

In [28]:
# 훈련 데이터와 레이블 데이터의 크기 확인해보기

print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터의 크기 : torch.Size([170, 10, 25])
레이블의 크기 : torch.Size([170, 10])


In [29]:
print(X[0])

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

In [30]:
print(Y[0])

tensor([24, 18, 15,  6, 22, 18, 23, 10, 12,  1])


### 2-2) 모델 구현하기
- 1-2의 모델과 유사하지만, 은닉층을 두개 쌓기

In [31]:
class Net(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, layers): # 현재 hidden_size는 dic_size와 같음.
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_dim, hidden_dim, num_layers=layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, hidden_dim, bias=True)

    def forward(self, x):
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

In [32]:
net = Net(dic_size, hidden_size, 2) # 이번에는 층을 두 개 쌓습니다.

- nn.RNN() 안에 num_layers라는 인자를 사용. 이는 은닉층을 몇 개 쌓을 것인지를 의미. 모델 선언 시 layers라는 인자에 2를 전달하여 은닉층을 두 개 쌓음. 비용 함수와 옵티마이저를 선언



In [33]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [34]:
# 입력을 넣어 출력 크기 확인해보기

outputs = net(X)
print(outputs.shape) # 3차원 텐서

torch.Size([170, 10, 25])


- (170, 10, 25)의 크기를 가지는데 각각 배치 차원, 시점(timesteps), 출력의 크기
- 나중에 정확도를 측정할 때는 이를 모두 펼쳐서 계산하게 되는데, 이때는 view를 사용하여 배치 차원과 시점 차원을 하나로 만듦



In [35]:
print(outputs.view(-1, dic_size).shape) # 2차원 텐서로 변환.

torch.Size([1700, 25])


In [36]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([170, 10])
torch.Size([1700])


In [37]:
# 옵티마이저와 손실 함수 정의

for i in range(100):
    optimizer.zero_grad()
    outputs = net(X) # (170, 10, 25) 크기를 가진 텐서를 매 에포크마다 모델의 입력으로 사용
    loss = criterion(outputs.view(-1, dic_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    # results의 텐서 크기는 (170, 10)
    results = outputs.argmax(dim=2)
    predict_str = ""
    for j, result in enumerate(results):
        if j == 0: # 처음에는 예측 결과를 전부 가져오지만
            predict_str += ''.join([char_set[t] for t in result])
        else: # 그 다음에는 마지막 글자만 반복 추가
            predict_str += char_set[result[-1]]

    print(predict_str)

gggg''g'llllegtl'l'gg'gl'g'lg't'lllll''g'lglll''gl'g'lllllllggll''gl'lelt't'l't't'lllet'glll'll'ggle'lg'''tltlglgl''gel'lllllll'lt'll'gggg'gllllg'gelllll''gl'ggllgll'gllllggllgeg'
                    b                                                  r                                    r                           r                                          
o tto to   to to  e t toe e t   o ton  to to   o  o       to to  e   to   to  t n t tot t  to   to  t t t ton t tot t n e   o  o to   to to   to  to  t   t   to  o  to to to  tte 
 wwwn wwn hwo lrn   s s n n s  le s n  to lo sw  lww   r  w  wo n  spwo luwo  no  n s n    s    wh  n s   w n n s n s n s ew n h s  w wo wr d wh wrw  so   n w o  nw    so      wn 
 heo lo  do hewodwe donlele ohele wo leoheoe doedwo loeloewoewo le do lelodhhedh,lo wo lhehbo doehdho woelohhlo whele dmo dhoelo wo lhohhwo do lheo eho doo wholehomohohholboehhodl
    r os e dhr o  e e tos e  r  r  r    hs o  e e  r   es e   h  e    os et r e d  s r e     r   o  